# World Cup 2026 Final Four Prediction
Predicts the winner among the 2026 semifinalists (France, Spain, England, Argentina)
using historical FIFA World Cup data (Fjelstul World Cup Database, 1930-2022).

**Method:** Poisson goal-scoring model with recency-weighted team attack/defense ratings,
simulated via Monte Carlo across the real knockout bracket.

## Step 1: Load data

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving tournament_standings.csv to tournament_standings.csv


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving tournaments.csv to tournaments.csv


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving team_appearances.csv to team_appearances.csv


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving matches.csv to matches.csv


In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = '.'

matches = pd.read_csv(f'{DATA_DIR}/matches.csv')
team_appearances = pd.read_csv(f'{DATA_DIR}/team_appearances.csv')
tournaments = pd.read_csv(f'{DATA_DIR}/tournaments.csv')
standings = pd.read_csv(f'{DATA_DIR}/tournament_standings.csv')

print(matches.shape, team_appearances.shape, tournaments.shape, standings.shape)

(900, 37) (1800, 36) (21, 18) (84, 7)


## Step 2: Inspect data (sanity check)

In [ ]:
print(team_appearances['stage_name'].unique())
print(tournaments[['tournament_id', 'year', 'winner']])

['group stage' 'semi-finals' 'final' 'round of 16' 'quarter-finals'
 'third-place match' 'final round' 'second group stage']
   tournament_id  year        winner
0        WC-1930  1930       Uruguay
1        WC-1934  1934         Italy
2        WC-1938  1938         Italy
3        WC-1950  1950       Uruguay
4        WC-1954  1954  West Germany
5        WC-1958  1958        Brazil
6        WC-1962  1962        Brazil
7        WC-1966  1966       England
8        WC-1970  1970        Brazil
9        WC-1974  1974  West Germany
10       WC-1978  1978     Argentina
11       WC-1982  1982         Italy
12       WC-1986  1986     Argentina
13       WC-1990  1990  West Germany
14       WC-1994  1994        Brazil
15       WC-1998  1998        France
16       WC-2002  2002        Brazil
17       WC-2006  2006         Italy
18       WC-2010  2010         Spain
19       WC-2014  2014       Germany
20       WC-2018  2018        France


## Step 3: Build recency-weighted team stats

More recent World Cups count more toward a team's rating than older ones,
since squads, tactics, and eras change a lot over 90+ years.
`HALF_LIFE` controls how fast old tournaments lose influence (in years).

In [ ]:
HALF_LIFE = 20
CURRENT_REF_YEAR = tournaments['year'].max()  # most recent tournament in the dataset

ta = team_appearances.merge(tournaments[['tournament_id', 'year']], on='tournament_id', how='left')
ta['weight'] = 0.5 ** ((CURRENT_REF_YEAR - ta['year']) / HALF_LIFE)

ta['weighted_goals_for'] = ta['goals_for'] * ta['weight']
ta['weighted_goals_against'] = ta['goals_against'] * ta['weight']

team_stats = ta.groupby('team_name').agg(
    matches_played=('match_id', 'count'),
    weighted_matches=('weight', 'sum'),
    weighted_goals_for=('weighted_goals_for', 'sum'),
    weighted_goals_against=('weighted_goals_against', 'sum'),
    total_goals_for=('goals_for', 'sum'),
    total_goals_against=('goals_against', 'sum'),
    wins=('win', 'sum'),
    draws=('draw', 'sum'),
    losses=('lose', 'sum'),
).reset_index()

league_avg_goals = ta['weighted_goals_for'].sum() / ta['weight'].sum()

team_stats['attack_rating'] = (team_stats['weighted_goals_for'] / team_stats['weighted_matches']) / league_avg_goals
team_stats['defense_rating'] = (team_stats['weighted_goals_against'] / team_stats['weighted_matches']) / league_avg_goals
team_stats['win_rate'] = team_stats['wins'] / team_stats['matches_played']

print(f"League-wide weighted avg goals per team-match: {league_avg_goals:.3f}")
team_stats.to_csv('team_strength_ratings.csv', index=False)
team_stats.sort_values('attack_rating', ascending=False).head(10)

League-wide weighted avg goals per team-match: 1.298


,team_name,matches_played,weighted_matches,weighted_goals_for,weighted_goals_against,total_goals_for,total_goals_against,wins,draws,losses,attack_rating,defense_rating,win_rate
33,Hungary,32,5.238171,12.346710,10.520597,87,57,15,3,14,1.816390,1.547741,0.468750
28,Germany,47,28.056598,55.776568,23.747767,95,48,32,5,10,1.531984,0.652267,0.680851
81,West Germany,62,14.099009,27.588132,15.640996,131,77,39,11,12,1.507895,0.854896,0.629032
8,Brazil,109,41.223343,78.439269,38.164973,229,105,76,14,19,1.466317,0.713443,0.697248
68,Soviet Union,31,6.839007,12.187920,7.437471,53,34,15,6,10,1.373327,0.838050,0.483871
45,Netherlands,50,24.620344,42.374193,20.479869,86,48,28,9,13,1.326309,0.641018,0.560000
58,Russia,14,10.640525,18.085425,14.520700,24,20,5,2,7,1.309794,1.051628,0.357143
27,France,66,30.320975,50.784083,26.602227,120,77,36,9,21,1.290690,0.676102,0.545455
17,Cuba,3,0.187500,0.312500,0.750000,5,12,1,1,1,1.284358,3.082460,0.333333
69,Spain,63,28.111971,46.241908,31.058551,99,72,31,11,21,1.267599,0.851388,0.492063


## Step 4: Check the Final Four's ratings

In [ ]:
FINAL_FOUR = ['France', 'Spain', 'England', 'Argentina']

print(team_stats[team_stats['team_name'].isin(FINAL_FOUR)][
    ['team_name', 'matches_played', 'win_rate', 'attack_rating', 'defense_rating']
].to_string(index=False))

team_name  matches_played  win_rate  attack_rating  defense_rating
Argentina              81  0.580247       1.222077        0.810988
  England              69  0.434783       0.998048        0.710983
   France              66  0.545455       1.290690        0.676102
    Spain              63  0.492063       1.267599        0.851388


## Step 5: Match simulation functions

Uses a Poisson model: each team's expected goals in a matchup = their attack rating
x opponent's defense rating x league average goals. Draws in knockout matches go
to a penalty-shootout coinflip, weighted slightly by attack rating.

In [ ]:
def get_ratings(team, strength_df):
    row = strength_df[strength_df['team_name'] == team].iloc[0]
    return row['attack_rating'], row['defense_rating']

def expected_goals(team_a, team_b, strength_df, league_avg):
    a_att, a_def = get_ratings(team_a, strength_df)
    b_att, b_def = get_ratings(team_b, strength_df)
    exp_a = a_att * b_def * league_avg
    exp_b = b_att * a_def * league_avg
    return exp_a, exp_b

def simulate_match(team_a, team_b, strength_df, league_avg, rng):
    exp_a, exp_b = expected_goals(team_a, team_b, strength_df, league_avg)
    goals_a = rng.poisson(exp_a)
    goals_b = rng.poisson(exp_b)
    if goals_a > goals_b:
        return team_a
    elif goals_b > goals_a:
        return team_b
    else:
        a_att, _ = get_ratings(team_a, strength_df)
        b_att, _ = get_ratings(team_b, strength_df)
        p_a = a_att / (a_att + b_att)
        return team_a if rng.random() < p_a else team_b

## Step 6: Monte Carlo simulation of the real bracket

Semifinals: France vs Spain, England vs Argentina -> Final -> Champion.
Simulated 50,000 times to get stable probabilities.

In [ ]:
rng = np.random.default_rng(42)
N_SIMS = 50000

SF1 = ('France', 'Spain')
SF2 = ('England', 'Argentina')

reach_final = {t: 0 for t in FINAL_FOUR}
win_title = {t: 0 for t in FINAL_FOUR}

for _ in range(N_SIMS):
    w1 = simulate_match(*SF1, team_stats, league_avg_goals, rng)
    w2 = simulate_match(*SF2, team_stats, league_avg_goals, rng)
    reach_final[w1] += 1
    reach_final[w2] += 1
    champion = simulate_match(w1, w2, team_stats, league_avg_goals, rng)
    win_title[champion] += 1

results = pd.DataFrame({
    'team': FINAL_FOUR,
    'reach_final_pct': [round(reach_final[t] / N_SIMS * 100, 2) for t in FINAL_FOUR],
    'win_world_cup_pct': [round(win_title[t] / N_SIMS * 100, 2) for t in FINAL_FOUR]
}).sort_values('win_world_cup_pct', ascending=False).reset_index(drop=True)

print(results.to_string(index=False))

     team  reach_final_pct  win_world_cup_pct
   France            57.59              33.79
Argentina            53.43              24.41
    Spain            42.41              22.11
  England            46.57              19.70


## Step 7: Export for Power BI

In [ ]:
results.to_csv('world_cup_2026_prediction.csv', index=False)
team_stats.to_csv('team_strength_ratings.csv', index=False)
ta.to_csv('team_appearances_with_weights.csv', index=False)

print("Exported: world_cup_2026_prediction.csv, team_strength_ratings.csv, team_appearances_with_weights.csv")

Exported: world_cup_2026_prediction.csv, team_strength_ratings.csv, team_appearances_with_weights.csv


## Notes / limitations

- Trained only on historical goal-scoring data through the last tournament in this dataset
  (does NOT know current squad form, injuries, or in-tournament momentum for 2026).
- Penalty shootout outcomes are approximated as a weighted coinflip, not modeled separately.
- To sharpen this model further: add confederation strength, host advantage, or
  leave-one-tournament-out validation against known past semifinalists to check accuracy.